# Clase 2 — Modelos Naif (versión local)

Mismo contenido que `z211_Naif.ipynb` del profe, adaptado para correr **local** (sin Colab).

La idea: 3 modelos tontos que son el **piso** que todo modelo serio tiene que ganar. Predecimos `tn` de **202002** para los 780 productos.

In [1]:
import polars as pl, os, sys
sys.path.insert(0, '/Users/giulialvaro/Library/CloudStorage/OneDrive-Personal/estudios/Austral/Materias/018-Laboratorio de Implementación III/labo3-2026ba/src')
from metrica import total_error_rate

DATA = r'/Users/giulialvaro/Library/CloudStorage/OneDrive-Personal/estudios/Austral/Materias/018-Laboratorio de Implementación III/labo3-2026ba/datasets'
os.makedirs('exp/naif', exist_ok=True)

## 1. Cargar y agregar a nivel producto × período
Colapso los clientes (sumo `tn`) y me quedo solo con los 780 productos a predecir.

In [2]:
d = pl.read_csv(f'{DATA}/sell-in.txt.gz', separator='\t')
apre = pl.read_csv(f'{DATA}/product_id_apredecir201912.txt', separator='\t')

v = (d.group_by('product_id','periodo').agg(pl.col('tn').sum().alias('tn'))
       .join(apre, on='product_id', how='inner')   # solo los 780
       .sort('product_id','periodo'))
print('filas:', v.height, '| productos:', v['product_id'].n_unique())
v.head()

filas: 22349 | productos: 780


product_id,periodo,tn
i64,i64,f64
20001,201701,934.77222
20001,201702,798.0162
20001,201703,1303.35771
20001,201704,1069.9613
20001,201705,1502.20132


## 2. Los 3 modelos Naif
- **último:** `tn(202002) = tn(201912)`
- **promedio 12m:** `= mean(tn[201901..201912])`
- **mismo mes año anterior:** `= tn(201902)` (si no existe, promedio)

In [3]:
ult  = v.filter(pl.col('periodo')==201912).select('product_id','tn')
prom = v.filter(pl.col('periodo').is_between(201901,201912)).group_by('product_id').agg(pl.col('tn').mean().alias('tn'))
feb  = v.filter(pl.col('periodo')==201902).select('product_id','tn')
mm   = (prom.join(feb, on='product_id', how='left', suffix='_f')
            .with_columns(pl.coalesce('tn_f','tn').alias('tn')).select('product_id','tn'))

ult.write_csv('exp/naif/naif_ultimo.csv')
prom.write_csv('exp/naif/naif_prom12.csv')
mm.write_csv('exp/naif/naif_mismomes.csv')
print('archivos de submit generados en exp/naif/')

archivos de submit generados en exp/naif/


## 3. Evaluar con el JUEZ (mundo ideal)
No puedo medir la métrica en los datos reales (no tengo la verdad de 202002), pero **sí en el mundo ideal** (`z262`), que tiene `tn_real`. Uso los mismos 3 naif ahí para ver cuál es mejor **sin gastar submits**.

In [4]:
vi = pl.read_csv(f'{DATA}/tb_ventas_ideal.csv')
ri = pl.read_csv(f'{DATA}/tb_realidad_ideal.csv')
u = vi['periodo'].max()

i_ult  = vi.filter(pl.col('periodo')==u).select('product_id','tn')
i_prom = vi.filter(pl.col('periodo')>=u-11).group_by('product_id').agg(pl.col('tn').mean().alias('tn'))
i_feb  = vi.filter(pl.col('periodo')==u-100).select('product_id','tn')
i_mm   = (i_prom.join(i_feb, on='product_id', how='left', suffix='_f')
              .with_columns(pl.coalesce('tn_f','tn').alias('tn')).select('product_id','tn'))

print('Total Error Rate (menor = mejor):')
print('  ultimo mes                :', round(total_error_rate(i_ult,  ri),4))
print('  promedio 12m              :', round(total_error_rate(i_prom, ri),4))
print('  mismo mes anio anterior   :', round(total_error_rate(i_mm,   ri),4))

Total Error Rate (menor = mejor):
  ultimo mes                : 0.6246
  promedio 12m              : 0.4357
  mismo mes anio anterior   : 0.4079


## Conclusión
El **mismo mes del año anterior** (0.4079) le gana al promedio (0.4357) y al último (0.6246): la **estacionalidad** ya aporta. Este es el número a batir con ARIMA / LightGBM.

👉 Para submitear a Kaggle: subir el CSV elegido de `exp/naif/` (columnas `product_id,tn`).